# Employee Attrition Analysis
**ABES 2nd Year B.Tech Data Analyst Project #10 | Domain: HR and education**

**Objective:** clean and explore the IBM HR Attrition dataset, measure attrition by department, tenure, salary band and gender, and prepare a verified clean file for Oracle SQL and Power BI.

**Dataset:** IBM HR Analytics Employee Attrition & Performance (Kaggle), `WA_Fn-UseC_-HR-Employee-Attrition.csv`, saved in `data/raw/`.

**Pipeline:** Raw CSV → *this notebook* (clean, test cases, bands, EDA) → `data/clean/hr_attrition_clean.csv` → Oracle (`sql/`) → Power BI dashboard.

| Section | Content |
|---|---|
| 1 | Setup and load |
| 2 | Data-quality checks |
| 3 | Cleaning |
| 4 | Test cases: missing tenure, salary outliers, single-employee departments |
| 5 | Feature engineering (bands) |
| 6 | KPIs |
| 7 | Exploratory analysis (attrition drivers) |
| 8 | Manual verification |
| 9 | Export clean data |
| 10 | Insights |

## 1. Setup and load

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RAW = Path('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
CLEAN = Path('../data/clean/hr_attrition_clean.csv')
SMALL_GROUP_THRESHOLD = 5

In [ ]:
raw = pd.read_csv(RAW, encoding='utf-8-sig')
print('Raw shape:', raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
# Standardise column names: 'YearsAtCompany' -> 'YEARS_AT_COMPANY' (matches the Oracle columns)
def to_snake_upper(name):
    s = re.sub(r'(?<!^)(?=[A-Z])', '_', name.strip()).upper()
    return {'OVER_TIME': 'OVERTIME'}.get(s, s)

df = raw.copy()
df.columns = [to_snake_upper(c) for c in df.columns]
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].str.strip()
list(df.columns)

## 2. Data-quality checks

In [ ]:
nulls = df.isna().sum()
print('Columns with nulls:', nulls[nulls > 0].to_dict() or 'none')
print('Fully duplicated rows:', df.duplicated().sum())
print('Duplicate EMPLOYEE_NUMBER:', df.duplicated(subset='EMPLOYEE_NUMBER').sum())

constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print('Constant columns (one value only):', constant_cols)

print('\nAttrition values:')
print(df['ATTRITION'].value_counts(dropna=False))

In [ ]:
df.describe().T[['min', 'max', 'mean']].round(1)

## 3. Cleaning
- Drop constant columns (`EMPLOYEE_COUNT`, `OVER18`, `STANDARD_HOURS`).
- Remove duplicate rows and duplicate employee IDs, keeping the first record.
- Keep only valid Attrition values (Yes/No), and create `ATTRITION_FLAG` (1 = left).

In [ ]:
rows_before = len(df)
df = df.drop(columns=constant_cols)
df = df.drop_duplicates()
df = df.drop_duplicates(subset='EMPLOYEE_NUMBER', keep='first')

df['ATTRITION'] = df['ATTRITION'].astype(str).str.strip().str.title()
invalid = ~df['ATTRITION'].isin(['Yes', 'No'])
print('Invalid attrition rows dropped:', int(invalid.sum()))
df = df[~invalid].copy()
df['ATTRITION_FLAG'] = (df['ATTRITION'] == 'Yes').astype(int)

for c in ['AGE', 'YEARS_AT_COMPANY', 'MONTHLY_INCOME', 'TOTAL_WORKING_YEARS']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
    df.loc[df[c] < 0, c] = np.nan

print(f'Rows: {rows_before} -> {len(df)}   Columns: {raw.shape[1]} -> {df.shape[1]}')

## 4. Test cases (from the project list)
### 4.1 Missing tenure
Rows with a missing `YEARS_AT_COMPANY` are **kept** and given `TENURE_BAND = 'Unknown'`, so the overall headcount stays correct.

In [ ]:
missing_tenure = df['YEARS_AT_COMPANY'].isna().sum()
inconsistent = (df['YEARS_AT_COMPANY'] > df['TOTAL_WORKING_YEARS']).sum()
print('Missing tenure rows:', missing_tenure)
print('Tenure > total working years (inconsistent):', inconsistent)

### 4.2 Salary outliers (IQR rule)
Outliers are **flagged, not removed**. High earners are real managers and directors, and removing them would bias the pay analysis.

In [ ]:
q1, q3 = df['MONTHLY_INCOME'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df['INCOME_OUTLIER_FLAG'] = ((df['MONTHLY_INCOME'] < low) | (df['MONTHLY_INCOME'] > high)).astype(int)
print(f'IQR bounds: {low:,.0f} to {high:,.0f}')
print('Outliers flagged:', df['INCOME_OUTLIER_FLAG'].sum())
print(df.loc[df['INCOME_OUTLIER_FLAG'] == 1, 'JOB_LEVEL'].value_counts().rename('outliers_by_job_level'))

plt.boxplot(df['MONTHLY_INCOME'].dropna(), vert=False)
plt.title('Monthly income distribution (IQR outliers shown as points)')
plt.xlabel('Monthly income')
plt.show()

### 4.3 Single-employee (small) departments and roles
A group with fewer than 5 employees gives an unreliable rate: one leaver in a group of one is 100% attrition.

In [ ]:
for col in ['DEPARTMENT', 'JOB_ROLE']:
    counts = df[col].value_counts()
    small = counts[counts < SMALL_GROUP_THRESHOLD]
    print(f'{col}: smallest group = {counts.min()} employees;',
          'small groups:', small.to_dict() if len(small) else 'none')
df['DEPARTMENT'].value_counts()

## 5. Feature engineering (bands)
| Band | Groups |
|---|---|
| Tenure | 0-2, 3-5, 6-10, 11+ years |
| Salary | <3K, 3K-6K, 6K-10K, 10K+ per month |
| Age | 18-25, 26-35, 36-45, 46-60 |

In [ ]:
TENURE_ORDER = ['0-2 yrs', '3-5 yrs', '6-10 yrs', '11+ yrs']
SALARY_ORDER = ['<3K', '3K-6K', '6K-10K', '10K+']
AGE_ORDER = ['18-25', '26-35', '36-45', '46-60']

df['TENURE_BAND'] = pd.cut(df['YEARS_AT_COMPANY'], [-0.1, 2, 5, 10, np.inf], labels=TENURE_ORDER).astype(object).fillna('Unknown')
df['SALARY_BAND'] = pd.cut(df['MONTHLY_INCOME'], [-0.1, 2999.99, 5999.99, 9999.99, np.inf], labels=SALARY_ORDER).astype(object).fillna('Unknown')
df['AGE_BAND'] = pd.cut(df['AGE'], [0, 25, 35, 45, 120], labels=AGE_ORDER).astype(object).fillna('Unknown')
df[['YEARS_AT_COMPANY', 'TENURE_BAND', 'MONTHLY_INCOME', 'SALARY_BAND', 'AGE', 'AGE_BAND']].head()

## 6. KPIs
**Attrition Rate % = Leavers / Total Employees × 100**

In [ ]:
total = len(df)
leavers = df['ATTRITION_FLAG'].sum()
kpis = pd.Series({
    'Total Employees': total,
    'Leavers': leavers,
    'Active Employees': total - leavers,
    'Attrition Rate %': round(leavers / total * 100, 2),
    'Avg Monthly Income': round(df['MONTHLY_INCOME'].mean(), 0),
    'Avg Tenure (yrs)': round(df['YEARS_AT_COMPANY'].mean(), 1),
    'Overtime Attrition Rate %': round(df.loc[df['OVERTIME'] == 'Yes', 'ATTRITION_FLAG'].mean() * 100, 2),
})
kpis.to_frame('value')

## 7. Exploratory analysis: what drives attrition?

In [ ]:
OVERALL = leavers / total * 100

def rate_table(col, order=None):
    g = df.groupby(col)['ATTRITION_FLAG'].agg(employees='count', leavers='sum')
    g['attrition_rate_pct'] = (g['leavers'] / g['employees'] * 100).round(2)
    if order:
        g = g.reindex([o for o in order if o in g.index] + [i for i in g.index if i not in order])
    return g

def plot_rate(col, title, order=None, horizontal=False):
    t = rate_table(col, order)
    ax = t['attrition_rate_pct'].plot(kind='barh' if horizontal else 'bar', color='#3B6E8F', rot=0)
    (ax.axvline if horizontal else ax.axhline)(OVERALL, color='#C0392B', ls='--', label=f'Company avg {OVERALL:.1f}%')
    ax.set_title(title)
    ax.set_ylabel('' if horizontal else 'Attrition rate %')
    ax.legend()
    plt.tight_layout()
    plt.show()
    return t

In [ ]:
plot_rate('DEPARTMENT', 'Attrition rate by department')

In [ ]:
plot_rate('TENURE_BAND', 'Attrition rate by tenure band', TENURE_ORDER)

In [ ]:
plot_rate('SALARY_BAND', 'Attrition rate by monthly salary band', SALARY_ORDER)

In [ ]:
plot_rate('OVERTIME', 'Attrition rate: overtime vs no overtime')

In [ ]:
plot_rate('GENDER', 'Attrition rate by gender')

In [ ]:
plot_rate('JOB_SATISFACTION', 'Attrition rate by job satisfaction (1 = Low, 4 = Very High)')

In [ ]:
plot_rate('AGE_BAND', 'Attrition rate by age band', AGE_ORDER)

In [ ]:
roles = rate_table('JOB_ROLE').sort_values('attrition_rate_pct', ascending=False)
roles['rank'] = roles['attrition_rate_pct'].rank(method='min', ascending=False).astype(int)
roles

In [ ]:
# Department x gender: what the Power BI department + gender slicers will show
pd.pivot_table(df, index='DEPARTMENT', columns='GENDER', values='ATTRITION_FLAG', aggfunc='mean').mul(100).round(2)

In [ ]:
# Department x tenure matrix (the Power BI drill-down matrix)
pd.pivot_table(df, index='DEPARTMENT', columns='TENURE_BAND', values='ATTRITION_FLAG', aggfunc='mean')[TENURE_ORDER].mul(100).round(2)

In [ ]:
# Pay gap: leavers vs stayers
print(df.groupby('ATTRITION')[['MONTHLY_INCOME', 'YEARS_AT_COMPANY', 'JOB_SATISFACTION']].agg(['mean', 'median']).round(1))

df.boxplot(column='MONTHLY_INCOME', by='ATTRITION', grid=False)
plt.suptitle('')
plt.title('Monthly income: stayed (No) vs left (Yes)')
plt.ylabel('Monthly income')
plt.show()

## 8. Manual verification
Recount the department and tenure attrition with a plain Python loop (no pandas `groupby`) and confirm that it matches the `groupby` result. Record the Excel COUNTIFS values in `docs/04_manual_verification.md`.

*Reference for the standard Kaggle file: 1,470 employees, 237 leavers, 16.12%.*

In [ ]:
def manual_rates(col):
    counts, left = {}, {}
    for value, attr in zip(df[col], df['ATTRITION']):
        counts[value] = counts.get(value, 0) + 1
        left[value] = left.get(value, 0) + (1 if attr == 'Yes' else 0)
    return {k: round(left[k] / counts[k] * 100, 2) for k in counts}

for col in ['DEPARTMENT', 'TENURE_BAND']:
    manual = pd.Series(manual_rates(col), name='manual_pct')
    auto = rate_table(col)['attrition_rate_pct'].rename('groupby_pct')
    check = pd.concat([manual, auto], axis=1)
    check['match'] = check['manual_pct'] == check['groupby_pct']
    display(check)
    assert check['match'].all(), f'Mismatch in {col}'

assert rate_table('DEPARTMENT')['employees'].sum() == total
print('Manual verification passed: loop counts match groupby, and department totals equal the overall total.')

## 9. Export the clean file
The column order **must** match `sql/02_load_staging.ctl` and `STG_HR_EMPLOYEE`.

In [ ]:
CLEAN_COLS = [
    'EMPLOYEE_NUMBER', 'AGE', 'AGE_BAND', 'GENDER', 'MARITAL_STATUS',
    'DEPARTMENT', 'JOB_ROLE', 'JOB_LEVEL', 'EDUCATION', 'EDUCATION_FIELD',
    'BUSINESS_TRAVEL', 'DISTANCE_FROM_HOME', 'OVERTIME',
    'MONTHLY_INCOME', 'SALARY_BAND', 'PERCENT_SALARY_HIKE', 'STOCK_OPTION_LEVEL',
    'TOTAL_WORKING_YEARS', 'YEARS_AT_COMPANY', 'TENURE_BAND', 'YEARS_IN_CURRENT_ROLE',
    'YEARS_SINCE_LAST_PROMOTION', 'YEARS_WITH_CURR_MANAGER', 'NUM_COMPANIES_WORKED',
    'TRAINING_TIMES_LAST_YEAR', 'ENVIRONMENT_SATISFACTION', 'JOB_SATISFACTION',
    'RELATIONSHIP_SATISFACTION', 'WORK_LIFE_BALANCE', 'JOB_INVOLVEMENT',
    'PERFORMANCE_RATING', 'ATTRITION', 'ATTRITION_FLAG', 'INCOME_OUTLIER_FLAG',
]
CLEAN.parent.mkdir(parents=True, exist_ok=True)
df[CLEAN_COLS].to_csv(CLEAN, index=False)
print(f'Saved {len(df)} rows x {len(CLEAN_COLS)} columns to {CLEAN}')

## 10. Insights
*Fill these in from the outputs above; each one is backed by a chart in Section 7 and in the dashboard.*

1. **Overtime:** employees who work overtime leave at __% vs __% for those who don't.
2. **Early tenure:** attrition is highest in the 0-2 year band (__%) and lowest at 11+ years (__%).
3. **Pay:** the <3K salary band has __% attrition. Leavers earn on average __ vs __ for stayers.
4. **Hotspots:** __ is the highest-attrition department (__%), and __ is the highest-attrition job role (__%).

**Recommendations:** limit sustained overtime, strengthen onboarding and mentoring in the first 2 years, and review pay for the lowest band and the highest-risk roles.

**Limitations:** a single fictional snapshot, so no time trend; the results show correlation, not causation.